# OpenMed DiseaseDetect SuperClinical 184M: fine-tuning on all 11 HoloKai entities

Same method as the PharmaDetect run: keep the encoder, replace the 1-class DISEASE head with a 23-tag BIO head (O + B/I for 11 entities), full fine-tuning on our original train set, best epoch chosen on validation.

Three learning rates (3e-5, 5e-5, 8e-5), 8 epochs, effective batch 16. V4 is scored **once**, at the end, for the validation-selected model only.

**Runtime → Change runtime type → T4 GPU**, then run the cells in order.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%pip install -q --only-binary=sentencepiece transformers==4.51.3 tokenizers==0.21.1 sentencepiece==0.2.2 safetensors==0.5.3 "setuptools<82"

In [ ]:
from pathlib import Path
import os, subprocess, sys
PROJECT = Path('/content/drive/MyDrive/Holokai_Project/week9/OpenMed-NER-DiseaseDetect-SuperClinical-184M')
assert PROJECT.is_dir(), f'Upload this folder to {PROJECT}'
ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')
def run(*args):
    """Run a project script and stream its output live."""
    p = subprocess.Popen([sys.executable, '-u', *map(str, args)], env=ENV, cwd=PROJECT,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout: print(line, end='')
    assert p.wait() == 0, f'Failed: {args}'
# synthetic self-tests (no data, no GPU)
run(PROJECT/'prepare_data.py', '--self-test')
run(PROJECT/'evaluate.py', '--self-test')

## 1. Prepare data (once)
Converts week6 train / val / V4 into BIO chunks under `data/`. Skipped if already prepared.

In [ ]:
if (PROJECT/'data'/'preparation_manifest.json').exists():
    print('Already prepared, skipping.')
else:
    run(PROJECT/'prepare_data.py')
run(PROJECT/'train.py', '--check-data')

## 2. Train three learning rates
About 20–30 min per group on a T4. Keep the tab open. Existing output folders are never overwritten; delete `outputs/<group>` to rerun one group.

In [ ]:
RUN_TRAINING = True
if RUN_TRAINING:
    run(PROJECT/'train.py')
else:
    print('Training disabled.')

In [ ]:
import json, pandas as pd
from IPython.display import display
rows = []
for group in ('learning_rate_3e-5', 'learning_rate_5e-5', 'learning_rate_8e-5'):
    path = PROJECT/'outputs'/group/'validation_results.json'
    if path.exists():
        r = json.loads(path.read_text())
        rows.append(dict(group=group, completed=r['completed'], best_epoch=r['best_epoch'], **{k: round(v, 4) if isinstance(v, float) else v for k, v in r['metrics']['overall'].items()}))
        print(group); display(pd.DataFrame(r['metrics']['per_label']).T.round(4))
display(pd.DataFrame(rows))
print('Reference, validation micro F1: our fine-tuned GLiNER 0.8733 (Final report 0.8746); PharmaDetect-434M best 0.8901')

## 3. Score V4 once (after all three groups are done)
Picks the group with the best validation F1, then evaluates only that model on V4. Refuses to run twice.

In [ ]:
RUN_V4 = True
if RUN_V4:
    run(PROJECT/'final_v4_eval.py')
    r = json.loads((PROJECT/'outputs'/'final_v4_results.json').read_text())
    display(pd.DataFrame(r['v4_metrics']['per_label']).T.round(4))
    print('Reference, V4 micro F1: our fine-tuned GLiNER 0.8825')